# Project 2: Large-Scale Sales Analytics & Windowed Business Intelligence

**Module**: 13_end_to_end_projects  
**Notebook**: 02_large_scale_sales_analytics.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Comprehensive sales analytics: monthly revenue trajectories, customer lifetime value (LTV), year-over-year growth, product ranking, and channel performance.
- **Why does this topic exist?**: Demonstrates the power of advanced DataFrame transformations, joins, and Window functions on realistic enterprise retail datasets.
- **What problem does it solve?**: Answering multi-dimensional executive dashboard questions efficiently at scale.
- **Where is it used in real-world Data Engineering?**: Powering executive dashboards and BI reporting models.

---

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Sales_Analytics_Project")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Extracting and Enriching Sales Dataset

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
customers = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/customers.csv")
products = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/products.csv")

# Denormalize
sales = (
    orders
    .filter("status = 'Completed'")
    .join(products, "product_id")
    .join(customers, "customer_id")
    .withColumn("order_month", F.date_format("order_ts", "yyyy-MM"))
    .withColumn("order_val", F.round(F.col("quantity") * F.col("unit_price") * (1.0 - F.col("discount")), 2))
)

sales.show(3)

## 4. Analysis 1: Top 5 Best-Selling Products per Category via DenseRank

In [ ]:
from pyspark.sql.window import Window

cat_win = Window.partitionBy("category").orderBy(F.desc("total_units"))

top_products = (
    sales
    .groupBy("category", "product_name")
    .agg(F.sum("quantity").alias("total_units"), F.round(F.sum("order_val"), 2).alias("total_rev"))
    .withColumn("rank", F.dense_rank().over(cat_win))
    .filter("rank <= 3")
    .orderBy("category", "rank")
)

top_products.show(15, truncate=False)

## 5. Analysis 2: Customer Lifetime Value (LTV) and Loyalty Tier Breakdown

In [ ]:
ltv_summary = (
    sales
    .groupBy("loyalty_tier", "segment")
    .agg(
        F.countDistinct("customer_id").alias("customer_count"),
        F.round(F.sum("order_val"), 2).alias("total_tier_revenue"),
        F.round(F.avg("order_val"), 2).alias("avg_order_value")
    )
    .orderBy(F.desc("total_tier_revenue"))
)

ltv_summary.show()

---

## Experiment: Month-over-Month Revenue Growth with Lag()

- **1. What are we testing?**: Compute monthly revenue and month-over-month growth percentage using F.lag().
- **2. What do we expect?**: Window function computes difference from prior month without self-join.

In [ ]:
monthly_rev = sales.groupBy("order_month").agg(F.round(F.sum("order_val"), 2).alias("revenue")).orderBy("order_month")
w_time = Window.orderBy("order_month")

mom_growth = (
    monthly_rev
    .withColumn("prev_month_rev", F.lag("revenue", 1).over(w_time))
    .withColumn("mom_growth_pct", F.round(((F.col("revenue") - F.col("prev_month_rev")) / F.col("prev_month_rev")) * 100, 2))
)

mom_growth.show(12)

**3. What happened?**: MoM growth calculated sequentially across order months.  
**4. Why did it happen?**: Window lag offset retrieved previous row values in sorted stream.  
**5. Practical Engineering Lesson**: Window lag() eliminates expensive self-joins in time-series growth queries.

---

## Practical Exercises

### Exercise 1 (Beginner): Channel Revenue Share
**Problem Statement**: Calculate total revenue and percentage share per channel.  
**Input / Context**: sales

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Regional Sales Pivot
**Problem Statement**: Pivot completed sales by region across product category.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Customer Repeat Purchase Rate
**Problem Statement**: Calculate percentage of customers with > 1 order.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Channel Revenue Share):
```python
tot = sales.select(F.sum('order_val')).collect()[0][0]
sales.groupBy('channel').agg(F.round(F.sum('order_val'), 2).alias('rev')).withColumn('pct', F.round(F.col('rev')/tot*100, 2)).show()
```

#### Solution 2 (Regional Sales Pivot):
```python
sales.groupBy('region').pivot('category').agg(F.round(F.sum('order_val'), 2)).show()
```

#### Solution 3 (Customer Repeat Purchase Rate):
```python
orders_per_cust = sales.groupBy('customer_id').count()
repeat_custs = orders_per_cust.filter('count > 1').count()
print(f'Repeat rate: {repeat_custs / orders_per_cust.count() * 100:.2f}%')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Analytical Mastery**: Window functions handle complex rankings and time offsets efficiently.
- **LTV Modeling**: Aggregations across dimensions reveal customer purchase behavior.

### Essential Technical Terms
- `LTV`: Customer Lifetime Value.
- `MoM Growth`: Month-over-month percentage change.

### Common Engineering Mistakes
- **Self-joins for time series**: Slow and error-prone; always prefer Window lag().

### Interview Questions on This Topic
- **Q: How do Window functions outperform self-joins for ranking?**
  - *A*: Window functions evaluate partitions in a single streaming pass without duplicating datasets or creating Cartesian comparisons.

### What to Study Next
Open 03_data_quality_pipeline.ipynb for automated data quality gates.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')